# 01 - Dataset Preprocessing & Herb Chunk Creation

**Bio-Heritage AI: RAG-Based Sri Lankan Herb Knowledge Assistant**

This notebook does Step 1 & Step 2 of the methodology:
1. Load the herb knowledge base CSV (18 real columns).
2. Clean it (remove duplicates, handle missing values).
3. Convert each herb record into a structured text chunk.
4. Save the chunks to `chunks/herb_chunks.json` for embedding in the next notebook.

> NOTE: We use the 18 columns that actually exist in the CSV. The README mentioned a few extra
> columns (source_name, source_url, verification_status, notes) that are NOT in the real file, so
> we ignore those. The real file has a single `source` column and a `source_type` column.

In [1]:
import pandas as pd
import numpy as np
import json
import os

# Paths (run this notebook from the project root or the notebooks/ folder)
CSV_PATH = "../data/sri_lankan_herb_knowledge_base.csv"
if not os.path.exists(CSV_PATH):
    CSV_PATH = "data/sri_lankan_herb_knowledge_base.csv"

CHUNKS_OUT = "../chunks/herb_chunks.json"
os.makedirs(os.path.dirname(CHUNKS_OUT), exist_ok=True)
print("Using CSV:", os.path.abspath(CSV_PATH))

Using CSV: c:\Users\THANUJA\OneDrive\Documents\Bio-Heritage-Ai\data\sri_lankan_herb_knowledge_base.csv


In [2]:
# Load the CSV. encoding='utf-8-sig' removes the BOM (the file starts with one).
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
print("Shape:", df.shape)
print("\nColumns (", len(df.columns), "):")
for c in df.columns:
    print(" -", c)
df.head(3)

Shape: (1550, 18)

Columns ( 18 ):
 - herb_id
 - herb_name_sinhala
 - herb_name_english
 - herb_name_latin
 - family
 - synonyms
 - treatment_for
 - parts_used_in_treatment
 - description
 - dosage
 - contraindications
 - compounds
 - native_distribution
 - conservation_status
 - edible_parts
 - medical_properties
 - source
 - source_type


,herb_id,herb_name_sinhala,herb_name_english,herb_name_latin,family,synonyms,treatment_for,parts_used_in_treatment,description,dosage,contraindications,compounds,native_distribution,conservation_status,edible_parts,medical_properties,source,source_type
0,SLH-0001,Gotukola,Asiatic pennywort,Centella asiatica,Apiaceae,NaN,memory improvement; wound healing; skin diseases,roots; leaves; flowers,"Asiatic pennywort (Centella asiatica), known i...","Dried roots powder: 3-5 g with warm water, twi...",May interact with antihypertensive medication,essential oils; phenolic acids; quercetin,Central highlands,Data Deficient,leaves (as mallum),antipyretic; antifungal,Sri Lanka Herbal Knowledge Survey Records,field_survey
1,SLH-0002,Kohomba,Neem,Azadirachta indica,Meliaceae,Kohomba kola; neem,skin diseases; fever; diabetes,flowers; leaves; bark,"Neem (Azadirachta indica), known in Sinhala as...",Standardised extract capsule: 500 mg once daily.,Prolonged high-dose use may cause loose stools...,alkaloids; lignans; phenolic acids; ellagic acid,Northern dry zone,Vulnerable,seeds (after processing),diuretic; antipyretic; antidiabetic,Traditional Ayurveda Compendium of Sri Lanka,traditional_text
2,SLH-0003,Iramusu,Indian sarsaparilla,Hemidesmus indicus,Apocynaceae,NaN,blood purification; urinary disorders; skin di...,tubers; fruits; tender shoots,"Indian sarsaparilla (Hemidesmus indicus), know...","Dried tubers powder: 3-5 g with warm water, on...",Use with caution in liver disease; May enhance...,alkaloids; saponins; lignans; gallic acid; ole...,Southern lowlands,Least Concern,young leaves,antibacterial; antioxidant; demulcent; hepatop...,Indigenous Medicine Field Notes Vol. II,traditional_text


In [3]:
# --- Basic data quality check ---
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate herb_id count:", df["herb_id"].duplicated().sum())
print("Duplicate full-row count:", df.duplicated().sum())

Missing values per column:
herb_id                      0
herb_name_sinhala            0
herb_name_english            0
herb_name_latin              0
family                       0
synonyms                   777
treatment_for                0
parts_used_in_treatment      0
description                  0
dosage                       0
contraindications            0
compounds                    0
native_distribution          0
conservation_status          0
edible_parts                 0
medical_properties           0
source                       0
source_type                  0
dtype: int64

Duplicate herb_id count: 0
Duplicate full-row count: 0


In [4]:
# --- Cleaning ---
# 1. Drop exact duplicate rows.
# 2. Fill missing text values with empty string so chunk-building never crashes.
# 3. Strip extra whitespace from every text cell.

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate rows. Remaining: {len(df)}")

text_cols = df.select_dtypes(include="object").columns
for c in text_cols:
    df[c] = df[c].fillna("").astype(str).str.strip()

df.head(3)

Dropped 0 duplicate rows. Remaining: 1550


C:\Users\THANUJA\AppData\Local\Temp\ipykernel_6852\3576565543.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns


,herb_id,herb_name_sinhala,herb_name_english,herb_name_latin,family,synonyms,treatment_for,parts_used_in_treatment,description,dosage,contraindications,compounds,native_distribution,conservation_status,edible_parts,medical_properties,source,source_type
0,SLH-0001,Gotukola,Asiatic pennywort,Centella asiatica,Apiaceae,,memory improvement; wound healing; skin diseases,roots; leaves; flowers,"Asiatic pennywort (Centella asiatica), known i...","Dried roots powder: 3-5 g with warm water, twi...",May interact with antihypertensive medication,essential oils; phenolic acids; quercetin,Central highlands,Data Deficient,leaves (as mallum),antipyretic; antifungal,Sri Lanka Herbal Knowledge Survey Records,field_survey
1,SLH-0002,Kohomba,Neem,Azadirachta indica,Meliaceae,Kohomba kola; neem,skin diseases; fever; diabetes,flowers; leaves; bark,"Neem (Azadirachta indica), known in Sinhala as...",Standardised extract capsule: 500 mg once daily.,Prolonged high-dose use may cause loose stools...,alkaloids; lignans; phenolic acids; ellagic acid,Northern dry zone,Vulnerable,seeds (after processing),diuretic; antipyretic; antidiabetic,Traditional Ayurveda Compendium of Sri Lanka,traditional_text
2,SLH-0003,Iramusu,Indian sarsaparilla,Hemidesmus indicus,Apocynaceae,,blood purification; urinary disorders; skin di...,tubers; fruits; tender shoots,"Indian sarsaparilla (Hemidesmus indicus), know...","Dried tubers powder: 3-5 g with warm water, on...",Use with caution in liver disease; May enhance...,alkaloids; saponins; lignans; gallic acid; ole...,Southern lowlands,Least Concern,young leaves,antibacterial; antioxidant; demulcent; hepatop...,Indigenous Medicine Field Notes Vol. II,traditional_text


In [5]:
# --- Build one text chunk per herb ---
# Each chunk combines the most useful fields into readable text.
# This text is what FAISS will search over, so we include names, uses,
# dosage, contraindications, compounds, properties, and the source.

def build_chunk_text(row):
    parts = []
    parts.append(f"Herb (Sinhala): {row['herb_name_sinhala']}")
    parts.append(f"Herb (English): {row['herb_name_english']}")
    parts.append(f"Latin name: {row['herb_name_latin']}")
    parts.append(f"Family: {row['family']}")
    if row['synonyms']:
        parts.append(f"Synonyms: {row['synonyms']}")
    parts.append(f"Used for (treatment): {row['treatment_for']}")
    parts.append(f"Parts used in treatment: {row['parts_used_in_treatment']}")
    parts.append(f"Description: {row['description']}")
    parts.append(f"Dosage: {row['dosage']}")
    parts.append(f"Contraindications: {row['contraindications']}")
    parts.append(f"Compounds: {row['compounds']}")
    parts.append(f"Medical properties: {row['medical_properties']}")
    parts.append(f"Native distribution: {row['native_distribution']}")
    parts.append(f"Conservation status: {row['conservation_status']}")
    parts.append(f"Edible parts: {row['edible_parts']}")
    parts.append(f"Source: {row['source']} ({row['source_type']})")
    return "\n".join(parts)

chunks = []
for _, row in df.iterrows():
    chunks.append({
        "herb_id": row["herb_id"],
        "herb_name_sinhala": row["herb_name_sinhala"],
        "herb_name_english": row["herb_name_english"],
        "herb_name_latin": row["herb_name_latin"],
        "treatment_for": row["treatment_for"],
        "dosage": row["dosage"],
        "contraindications": row["contraindications"],
        "medical_properties": row["medical_properties"],
        "source": row["source"],
        "source_type": row["source_type"],
        "chunk_text": build_chunk_text(row)
    })

print("Total chunks:", len(chunks))
print("\n--- Example chunk_text ---\n")
print(chunks[0]["chunk_text"])

Total chunks: 1550

--- Example chunk_text ---

Herb (Sinhala): Gotukola
Herb (English): Asiatic pennywort
Latin name: Centella asiatica
Family: Apiaceae
Used for (treatment): memory improvement; wound healing; skin diseases
Parts used in treatment: roots; leaves; flowers
Description: Asiatic pennywort (Centella asiatica), known in Sinhala as Gotukola, is a medicinal plant of the family Apiaceae recorded in Sri Lankan traditional medicine. It is traditionally used for memory improvement, wound healing, skin diseases. The roots are the most commonly used part, usually prepared as a powder. Reported properties include antipyretic, antifungal activity.
Dosage: Dried roots powder: 3-5 g with warm water, twice daily.
Contraindications: May interact with antihypertensive medication
Compounds: essential oils; phenolic acids; quercetin
Medical properties: antipyretic; antifungal
Native distribution: Central highlands
Conservation status: Data Deficient
Edible parts: leaves (as mallum)
Source: 

In [6]:
# --- Save chunks to JSON ---
with open(CHUNKS_OUT, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Saved {len(chunks)} chunks to {os.path.abspath(CHUNKS_OUT)}")
print("\nDay 1 done. Next: 03_faiss_index_creation.ipynb to embed these chunks.")

Saved 1550 chunks to c:\Users\THANUJA\OneDrive\Documents\Bio-Heritage-Ai\chunks\herb_chunks.json

Day 1 done. Next: 03_faiss_index_creation.ipynb to embed these chunks.
